## Import librairies

In [26]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.2f}'.format

In [27]:
def compute_backtest(days_to_maturity, strike_pct, price_ts, option_type, sigma):
    price_ts.name = 'F'
    # build cyclic count 5->0 and, after each 0, insert a duplicated row that starts the next cycle at 5
    orig = price_ts.reset_index()
    idx_col = orig.columns[0]  # original index column name (usually the timestamp)
    rows = []
    count = days_to_maturity

    for _, r in orig.iterrows():
        d = r.to_dict()
        d['days_to_maturity'] = count
        rows.append(d)
        if count == 0:
            # duplicate the same row but set count to 5 to start the next cycle
            dup = r.to_dict()
            dup['days_to_maturity'] = days_to_maturity
            rows.append(dup)
            count = days_to_maturity-1  # next appended original row should get 4
        else:
            count -= 1

    bt_df = pd.DataFrame(rows).set_index(idx_col)

    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'F0'] = bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'F']
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'strike_date'] = bt_df.loc[bt_df['days_to_maturity']==days_to_maturity].index
    bt_df = bt_df.ffill()
    bt_df['K'] = bt_df['F0'] * strike_pct
    bt_df['T'] = bt_df['days_to_maturity'] / 252

    rows = []
    for index, row in bt_df.iterrows():
        
        row_dict = row.to_dict()

        row_dict['date'] = index
        F = row_dict['F']
        K = row_dict['K']
        T = row_dict['T']

        pricing_results = BSMModel.compute_option_with_forward(
            F=F,
            K=K,
            T=T,
            r=0,
            sigma=sigma,
            option_type=option_type,
            compute_greeks=True
            )
        merged_dict = {**row_dict, **pricing_results}
        rows.append(merged_dict)
    bt_df = pd.DataFrame(rows)
    bt_df['dP'] = bt_df['price'].diff()
    bt_df['dH'] = bt_df['F'].diff() * bt_df['delta'].shift(1)

    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'dP'] = 0 
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'dH'] = 0

    # Extra outputs
    bt_df['dP_cumsum'] = bt_df['dP'].cumsum()
    bt_df['dH_cumsum'] = bt_df['dH'].cumsum()
    return bt_df

## Inputs

In [28]:
long_asset_symbol = 'SPY'
long_asset_vol = 0.10

short_asset_symbol = 'QQQ'
short_asset_vol = 0.10

vol_window = 20
beta_window = 20

option_type = 'call'
target_delta = 0.1
target_slide = 0.3
slide_list = [target_slide, 0.2, 0.1, 0.05]

beta_type = 'override' # 'mean', 'quantile' or 'override'
override_value = 0.45
quantile = 10

## Import data

In [29]:
long_assset_df = pd.read_csv(f'data/{long_asset_symbol}.csv', index_col=0, parse_dates=True)['price']
short_asset_df = pd.read_csv(f'data/{short_asset_symbol}.csv', index_col=0, parse_dates=True)['price']
long_assset_df.name = long_asset_symbol
short_asset_df.name = short_asset_symbol
df = pd.concat([long_assset_df, short_asset_df], axis=1)

## Volatility Target

In [30]:
vol_target = pd.Series(index=[long_asset_symbol, short_asset_symbol], data=[long_asset_vol, short_asset_vol], name='target_vol')
df_log_returns = np.log(df / df.shift(1))
df_rolling_std = df_log_returns.rolling(window=vol_window).std() * np.sqrt(252)
df_leverage = vol_target / df_rolling_std
df_leverage = df_leverage.dropna()
df = df.loc[df_leverage.index]
df_vt = (df.pct_change() * df_leverage.shift(1)).fillna(0).add(1).cumprod() * 100
df_vt_change = df_vt.pct_change()
df_vt_level = df_vt_change.fillna(0).add(1).cumprod() * 100

## Beta calibration

In [31]:
df_vt_roll_returns = df_vt_level.rolling(window=beta_window).apply(lambda x: x.iloc[-1]/x.iloc[0]-1).dropna()
beta_ts = df_vt_roll_returns[long_asset_symbol] / df_vt_roll_returns[short_asset_symbol]

df_regression = df_vt_roll_returns[[long_asset_symbol, short_asset_symbol]].copy()
df_regression['beta'] = beta_ts

long_asset_K = BSMModel.solve_delta_strike(F=100, T=beta_window/252, sigma=long_asset_vol, r=0, option_type=option_type, target_delta=target_delta)
short_asset_K = BSMModel.solve_delta_strike(F=100, T=beta_window/252, sigma=short_asset_vol, r=0, option_type=option_type, target_delta=target_delta)
short_asset_threshold = short_asset_K / 100 -1

if option_type == 'put':
    cond_serie = df_regression[df_regression[short_asset_symbol] < short_asset_threshold]['beta']
else:
    cond_serie = df_regression[df_regression[short_asset_symbol] > short_asset_threshold]['beta']

if beta_type == 'mean':
    cond_beta = cond_serie.mean()
elif beta_type == 'quantile':
    cond_beta = np.percentile(cond_serie, quantile)
elif beta_type == 'override':
    cond_beta = override_value

print(f'Beta: {cond_beta:.2f}')

Beta: 0.45


## Backtesting

In [32]:
pricing = dict()

beta_slide = np.round(target_slide*cond_beta, 2)
slide_list += [beta_slide]

long_asset_out = BSMModel.compute_option(F=100, K=long_asset_K, T=beta_window/252, r=0, sigma=long_asset_vol, option_type=option_type, compute_greeks=True, slide_list=slide_list)
short_asset_out = BSMModel.compute_option(F=100, K=short_asset_K, T=beta_window/252, r=0, sigma=short_asset_vol, option_type=option_type, compute_greeks=True, slide_list=slide_list)

long_asset_out['moneyness'] = long_asset_K / 100
short_asset_out['moneyness'] = short_asset_K / 100

long_asset_out['target_slide'] = long_asset_out[beta_slide]
short_asset_out['target_slide'] = short_asset_out[target_slide]

pricing[long_asset_symbol] = long_asset_out
pricing[short_asset_symbol] = short_asset_out

pricing_df = pd.DataFrame(pricing).transpose()
pricing_df['qty'] = 100_000_000 / pricing_df['target_slide']
pricing_df['delta_cash'] = pricing_df['delta'] * 100
for key in ['price', 'delta_cash', 'gamma', 'vega', 'theta'] + slide_list:
    pricing_df[f'ccy_{key}'] = pricing_df[key] * pricing_df['qty']
cols = list(filter(lambda x: str(x).startswith('ccy'), pricing_df.columns))
display(pricing_df[cols])
display(pricing_df[cols].multiply(pd.Series(index=[long_asset_symbol, short_asset_symbol], data=[1,-1]), axis=0).sum())

,ccy_price,ccy_delta_cash,ccy_gamma,ccy_vega,ccy_theta,ccy_0.3,ccy_0.2,ccy_0.1,ccy_0.05,ccy_0.14
SPY,"1,299,024.45","98,532,957.79","613,777.63","487,125.10","-121,781.28","257,610,971.80","159,102,174.15","60,791,719.77","17,656,311.53","100,000,000.00"
QQQ,"504,258.20","38,248,742.71","238,257.56","189,093.31","-47,273.33","100,000,000.00","61,760,635.83","23,598,264.99","6,853,866.28","38,818,222.41"


ccy_price            794,766.25
ccy_delta_cash    60,284,215.08
ccy_gamma            375,520.06
ccy_vega             298,031.80
ccy_theta            -74,507.95
ccy_0.3          157,610,971.80
ccy_0.2           97,341,538.32
ccy_0.1           37,193,454.78
ccy_0.05          10,802,445.25
ccy_0.14          61,181,777.59
dtype: float64

In [33]:
504258.20/1299024.45

0.3881822239758459

In [34]:
long_asset_payoff = df_vt_level[long_asset_symbol].rolling(window=beta_window).apply(lambda x: x.iloc[-1]/x.iloc[0]-1)
long_asset_payoff = long_asset_payoff.to_frame(name='returns')
long_asset_payoff['ST'] = 100 * (1+long_asset_payoff['returns'])
long_asset_payoff['K'] = long_asset_K
if option_type == 'call':
    long_asset_payoff['payoff'] = (long_asset_payoff['ST']-long_asset_payoff['K']).clip(lower=0)
else:
    long_asset_payoff['payoff'] = (long_asset_payoff['K']-long_asset_payoff['ST']).clip(lower=0)
long_asset_payoff['total_pnl'] = long_asset_payoff['payoff'] - pricing_df.loc[long_asset_symbol, 'price']
long_asset_payoff['ccy_total_pnl'] = long_asset_payoff['total_pnl'] * pricing_df.loc[long_asset_symbol, 'qty']

short_asset_payoff = df_vt_level[short_asset_symbol].rolling(window=beta_window).apply(lambda x: x.iloc[-1]/x.iloc[0]-1)
short_asset_payoff = short_asset_payoff.to_frame(name='returns')
short_asset_payoff['ST'] = 100 * (1+short_asset_payoff['returns'])
short_asset_payoff['K'] = short_asset_K
if option_type == 'call':
    short_asset_payoff['payoff'] = (short_asset_payoff['ST']-short_asset_payoff['K']).clip(lower=0)
else:
    short_asset_payoff['payoff'] = (short_asset_payoff['K']-short_asset_payoff['ST']).clip(lower=0)

short_asset_payoff['total_pnl'] = short_asset_payoff['payoff'] - pricing_df.loc[short_asset_symbol, 'price']
short_asset_payoff['ccy_total_pnl'] = short_asset_payoff['total_pnl'] * pricing_df.loc[short_asset_symbol, 'qty']

x = (long_asset_payoff-short_asset_payoff)['ccy_total_pnl'].fillna(0)

## Plot

In [35]:
px.bar(x.groupby(x.index.year).sum())

In [36]:
x_cumsum = x.cumsum()
fig = px.line(x_cumsum)
for year in x.index.year.unique():
    fig.add_vline(x=pd.Timestamp(year=year, month=1, day=1), line_dash="dot", line_color="red")
fig.show()

In [37]:
long_bt = compute_backtest(
    days_to_maturity=beta_window, 
    strike_pct=pricing_df.loc[long_asset_symbol, 'moneyness'], 
    price_ts=df_vt_level[long_asset_symbol].copy(), 
    option_type=option_type, 
    sigma=long_asset_vol)

short_bt = compute_backtest(
    days_to_maturity=beta_window, 
    strike_pct=pricing_df.loc[short_asset_symbol, 'moneyness'], 
    price_ts=df_vt_level[short_asset_symbol].copy(), 
    option_type=option_type, 
    sigma=short_asset_vol)

long_bt = long_bt.groupby('strike_date')['dH'].sum() * (100 / long_bt.groupby('strike_date')['F0'].mean())
short_bt = short_bt.groupby('strike_date')['dH'].sum() * (100 / short_bt.groupby('strike_date')['F0'].mean())

In [38]:
long_bt = long_bt.cumsum()
short_bt = short_bt.cumsum()

In [39]:
x = long_bt * pricing_df.loc[long_asset_symbol, 'qty'] - short_bt * pricing_df.loc[short_asset_symbol, 'qty']

In [40]:
px.line(x)

In [41]:
px.scatter(cond_serie)

In [42]:
df_regression.sort_values(by=short_asset_symbol).head(10)

,SPY,QQQ,beta
2020-03-16,-0.10,-0.09,1.18
2020-03-17,-0.10,-0.08,1.19
2020-03-12,-0.10,-0.08,1.24
2020-03-18,-0.10,-0.08,1.25
2015-08-25,-0.08,-0.08,1.04
2008-10-09,-0.07,-0.08,0.94
2018-10-29,-0.10,-0.08,1.33
2019-06-03,-0.06,-0.08,0.82
2015-08-24,-0.07,-0.08,0.96
2008-10-08,-0.06,-0.08,0.78
